# IOAI — 2025 Team Selection Day4 Nlp Masked Word (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day4-nlp-masked-word'
for f in ['train.csv','val.csv','public_test.csv','sample_submission.csv']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train.csv','val.csv','public_test.csv','sample_submission.csv'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Day 4 — Masked-Word Position · 모범답안 (원본 대회 코드)

> **Kazakhstan – Team Selection 2025 · Day 4 (NLP / Transformers)**

이 노트북은 Batyr Yerdenov 의 **실제 TST 제출 코드**(github.com/batyrq/ioai-tst-kazakhstan)를 거의 **그대로** 옮긴 것입니다.
데이터는 공개 up-solving 미러(`tst-day-4-upsolving`)의 원본을 동봉본으로 사용하며, 바뀐 부분은 **데이터 경로**와 **로컬 채점용 검증 예측 추가**뿐입니다(주석으로 표시). 더 정돈·향상된 재구현은 **모범답안2** 탭을 참고하세요.


*주의: 원본 하이퍼파라미터(EPOCHS=10, 전체 데이터)를 그대로 둬 학습이 무겁습니다. 빠른 채점용 경량 버전은 모범답안2.*

In [ ]:
# --- 데이터 경로 어댑터(추가): 동봉 데이터 폴더로 이동 (원본 코드는 그대로) ---
# 현재 폴더(".")를 먼저 확인 → 모범답안 작업본(_solution)에 머물러 제출물이 거기에 저장되게 한다
# (상위 폴더에도 데이터가 풀려 있어 "."보다 ".."를 먼저 보면 self(내 풀이) 폴더에 잘못 저장됨).
import os
for _b in [".", "..", "../.."]:
    if os.path.exists(os.path.join(_b, "train.csv")):
        os.chdir(_b); break
print("data dir:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW
import random

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

tqdm.pandas()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# === Очистка текста ===
BAD_CHARS = set(['"', "'", '“', '”', '‘', '’', '«', '»', '‹', '›', '„', '‟', '‚', '`',
                 '\u200b', '\u200c', '\u200d', '\ufeff', '\u202a', '\u202b',
                 '\u202c', '\u202d', '\u202e'])

def is_valid_char(c):
    if c in [' ', '.', ',']:
        return True
    if c in BAD_CHARS:
        return False
    name = unicodedata.name(c, '')
    return 'CYRILLIC' in name

def clean_text(text):
    text = str(text)
    cleaned = ''.join(c for c in text if is_valid_char(c))
    cleaned = re.sub(r'[.,]{2,}', '.', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def is_clean_sentence(sent):
    sent = sent.strip()
    words = sent.split()
    if len(words) < 5 or len(words) > 25:
        return False
    if sent.isupper():
        return False
    if sum(1 for c in sent if c.isupper()) > len(sent) * 0.4:
        return False
    if re.search(r'(.)\1{2,}', sent):
        return False
    if re.match(r'^[А-Яа-я]{1,2}(\s|$)', sent):
        return False
    if len(re.findall(r'[аеиоуыіұүәө]', sent, flags=re.IGNORECASE)) < 3:
        return False
    return True

# === Загрузка данных ===
df_raw = pd.read_csv("train.csv")
test_df = pd.read_csv("public_test.csv")

sentences = []
for row in tqdm(df_raw["sentence"]):
    cleaned = clean_text(row)
    parts = re.split(r'[.!?]', cleaned)
    for part in parts:
        part = part.strip()
        if is_clean_sentence(part):
            sentences.append(part)

clean_df = pd.DataFrame({"clean_sentence": list(set(sentences))})
print(f"✅ Предобработка завершена: {len(clean_df)} предложений")

# === Генерация обучающих примеров (по 3 маски) ===
def generate_n_masks(sentence, n=3):
    words = sentence.split()
    if len(words) < n + 1:
        return []
    idxs = np.random.choice(len(words), size=n, replace=False)
    examples = []
    for i in idxs:
        masked = " ".join(words[:i] + words[i+1:])
        examples.append((masked, i))
    return examples

train_sentences, val_sentences = train_test_split(clean_df["clean_sentence"], test_size=0.1, random_state=42)

train_data = []
for sent in tqdm(train_sentences):
    train_data.extend(generate_n_masks(sent, n=3))

val_data = []
for sent in tqdm(val_sentences):
    val_data.extend(generate_n_masks(sent, n=3))

train_df = pd.DataFrame(train_data, columns=["masked_sentence", "label"])
val_df = pd.DataFrame(val_data, columns=["masked_sentence", "label"])

# === Токенизатор и модель ===
tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")

class MaskedSentenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=32):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = self.tokenizer(row["masked_sentence"], return_tensors="pt", 
                                 max_length=self.max_len, padding="max_length", 
                                 truncation=True)
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        label = torch.tensor(row["label"])
        return input_ids, attention_mask, label

train_ds = MaskedSentenceDataset(train_df, tokenizer)
val_ds = MaskedSentenceDataset(val_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

class BertPositionModel(nn.Module):
    def __init__(self, bert_name="bert-base-multilingual-cased", hidden_size=768):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_size, 25)  # Предположим максимум 25 слов

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        return self.classifier(cls)

model = BertPositionModel().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

# === Обучение ===
EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for input_ids, attention_mask, labels in tqdm(train_loader):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"📚 Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")

    # Валидация
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in val_loader:
            input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
            logits = model(input_ids, attention_mask)
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            true.extend(labels.numpy())
    acc = accuracy_score(true, preds)
    print(f"🎯 Validation Accuracy: {acc:.4f}")

# === Инференс на тесте ===
test_df["masked_sentence_clean"] = test_df["masked_sentence"].progress_apply(clean_text)

class TestDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=32):
        self.sentences = df["masked_sentence_clean"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        encoded = self.tokenizer(self.sentences[idx], return_tensors="pt",
                                 padding="max_length", truncation=True,
                                 max_length=self.max_len)
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        return input_ids, attention_mask

test_ds = TestDataset(test_df, tokenizer)
test_loader = DataLoader(test_ds, batch_size=64)

model.eval()
all_preds = []
with torch.no_grad():
    for input_ids, attention_mask in tqdm(test_loader):
        input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
        logits = model(input_ids, attention_mask)
        pred = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(pred)

# Эвристика: если начало с маленькой буквы — позиция = 0
def is_lower_start(s):
    s = s.strip()
    return len(s) > 0 and s[0].islower()

test_df["start_lower"] = test_df["masked_sentence_clean"].apply(is_lower_start)
final_preds = np.where(test_df["start_lower"], 0, all_preds)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "word_index": final_preds
})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv сохранён")


In [ ]:

# === [추가] 로컬 채점용: 동봉 검증셋(val.csv) 예측 → submission_val.csv ===
our_val = pd.read_csv("val.csv")                      # ID, masked_sentence
our_val["masked_sentence_clean"] = our_val["masked_sentence"].apply(clean_text)
ov_ds = TestDataset(our_val, tokenizer); ov_loader = DataLoader(ov_ds, batch_size=64)
model.eval(); ov_preds = []
with torch.no_grad():
    for input_ids, attention_mask in ov_loader:
        input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
        ov_preds.extend(model(input_ids, attention_mask).argmax(dim=1).cpu().numpy())
our_val["start_lower"] = our_val["masked_sentence_clean"].apply(is_lower_start)
ov_final = np.where(our_val["start_lower"], 0, ov_preds)
import pandas as _pd
_pd.DataFrame({"ID": our_val["ID"], "word_index": ov_final}).to_csv("submission_val.csv", index=False)
print("saved submission_val.csv", len(ov_final))


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_val.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)